In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

!pwd
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')
!pwd
# Get the current working directory
current_directory = os.getcwd()

# Specify the file name
file_name = "mastershifted.txt"

# Construct the full path to the file
file_path = os.path.join(current_directory, file_name)

# Open the file
try:
    with open(file_path, "r") as f:
        # Do something with the file (e.g., print its contents)
        print("Loaded successfully")
except FileNotFoundError:
    print(f"The file '{file_name}' was not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")

/content
/content/drive/My Drive/Colab Notebooks
Loaded successfully


In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Reshape, LSTM, Dense, Dropout, Activation, TimeDistributed, Flatten, BatchNormalization, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt

In [ ]:
colnames=( 'Bx', 'By', 'Bz', 'Vx', 'Density', 'Labels')
df = pd.read_csv('mastershifted.txt', sep='\s+', names=colnames)
dfc=df.drop(['Labels'], axis=1)

n_timesteps = 120

Xcols=[x for x in dfc.columns if x!= 'Labels']
n_features=len(Xcols)
model= Sequential()
X = df[Xcols].values
y = np.array([0, 1] * (76292 // 2))  # Alternating 0s and 1s for balanced labels

In [ ]:
# Number of splits (folds)
k = 5  # You can adjust k based on your preference

# Initialize StratifiedKFold without shuffling to preserve temporal order
skf = StratifiedKFold(n_splits=k, shuffle=False)

# List to store the results for each fold (e.g., validation accuracy)
test_accuracies = []

In [ ]:
scaler = MinMaxScaler() #standardscaler before
X = X.reshape(76292, 120, 5)

In [ ]:
from sklearn.metrics import f1_score
f1_scores = []

In [ ]:
# Stratified K-Fold loop
for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"Fold {fold + 1}")

    # Split the data into training and validation sets
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    # Further split the validation set to create the test set (50% of the fold's data for test)
    val_size = len(X_val) // 2  # 50% of the validation set to create test set
    X_val, X_test = X_val[:val_size], X_val[val_size:]
    y_val, y_test = y_val[:val_size], y_val[val_size:]

    # Now we have X_train, X_val, X_test, and corresponding labels for each fold

    # Reshape for scaling: (samples, time_steps, features, channels) to (samples * time_steps, features)
    X_train_reshaped = X_train.reshape(-1, 5)  # Shape: (samples * time_steps, features)
    X_val_reshaped = X_val.reshape(-1, 5)  # Shape: (samples * time_steps, features)
    X_test_reshaped = X_test.reshape(-1, 5)  # Shape: (samples * time_steps, features)

    # Fit the scaler on the training data and transform both the training, validation, and test data
    X_train_scaled = scaler.fit_transform(X_train_reshaped)
    X_val_scaled = scaler.transform(X_val_reshaped)
    X_test_scaled = scaler.transform(X_test_reshaped)

    # Reshape back to the original shape (samples, time_steps, features, channels)
    X_train_scaled = X_train_scaled.reshape(X_train.shape[0], 120, 5)
    X_val_scaled = X_val_scaled.reshape(X_val.shape[0], 120, 5)
    X_test_scaled = X_test_scaled.reshape(X_test.shape[0], 120, 5)

#-----------------------------------------------------------------------------------------------------
    model = Sequential()
    model.add(LSTM(32, return_sequences=False))
    model.add(BatchNormalization())
    model.add(Dense(32, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])

    history=model.fit(X_train_scaled, y_train, epochs=45, batch_size=128, validation_data=(X_val_scaled, y_val))
#-----------------------------------------------------------------------------------------------------
    # Evaluate the model on the test data
    test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
    test_accuracies.append(test_accuracy)
    print(f'Test Accuracy for Fold {fold + 1}: {test_accuracy:.4f}')

    y_pred = model.predict(X_test_scaled)
    y_pred = (y_pred > 0.5).astype(int)  # Convert probabilities to binary predictions #USE 0.5!!!!!!!!!!!!

    fold_f1_score = f1_score(y_test, y_pred)
    f1_scores.append(fold_f1_score)
    print(f'F1 Score for Fold {fold + 1}: {fold_f1_score:.4f}')

    # Optionally, clear the Keras session to avoid memory issues during repeated training
    K.clear_session()

    # Store validation accuracy (optional)
    #test_accuracies.append(history.history['test_accuracy'][-1])



# Calculate the average test accuracy across all folds (optional)
avg_test_accuracy = np.mean(test_accuracies)
print(f'Average test Accuracy: {avg_test_accuracy:.4f}')

avg_f1_score = np.mean(f1_scores)
print(f'Average F1 Score: {avg_f1_score:.4f}')
